# Object Detection System - SSD Fine-Tuning Lab
### Capstone Project: Computer Vision Transfer Learning with TensorFlow and OpenCV

Welcome! This notebook guides you through building a custom **Object Detection System**. You will learn how to fine-tune the output classification layer of a pre-trained **Single Shot MultiBox Detector (SSD)** model on a custom subset of **Kaggle's Open Images Object Detection Dataset**.

#### Learning Objectives:
1. Understand the Single Shot MultiBox Detector (SSD) architecture.
2. Fetch and filter a custom subset of the Open Images dataset.
3. Implement transfer learning in TensorFlow/Keras to replace and train the classification head.
4. Construct a custom loss function (MultiBox Loss) merging categorization and bounding-box regression.
5. Perform model inference and draw bounding boxes using OpenCV.

## Step 1: Environment Setup
First, we install and import the required dependencies. If you are running on Google Colab, TensorFlow and OpenCV are pre-installed. We will also install `pandas` to easily process annotations.

In [ ]:
# Install dependencies if not present
# !pip install tensorflow opencv-python pandas matplotlib pillow requests

In [ ]:
import os
import urllib.request
import json
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from PIL import Image

print("TensorFlow Version:", tf.__version__)
print("OpenCV Version:", cv2.__version__)

## Step 2: Open Images Dataset Subset Fetcher
Google's **Open Images Dataset** contains millions of images annotated with bounding boxes. Downloading the full dataset requires terabytes of storage. 

To make it manageable, we download the official **Validation Annotations** and filter for 5 specific classes of everyday items using their unique Machine IDs (MIDs):
- Person: `/m/01g317`
- Laptop: `/m/03b443`
- Chair: `/m/01mzpv`
- Book: `/m/01dx8v`
- Table: `/m/01y9k5`

In [ ]:
# Define dataset directories
DATASET_DIR = "dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

# Target classes MIDs mapping
TARGET_CLASSES = {
    "/m/01g317": "person",
    "/m/03b443": "laptop",
    "/m/01mzpv": "chair",
    "/m/01dx8v": "book",
    "/m/01y9k5": "table"
}

# URLs for validation data split
VAL_BOX_URL = "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv"
VAL_METADATA_URL = "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv"

def download_file(url, filename):
    filepath = os.path.join(DATASET_DIR, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
    else:
        print(f"{filename} already exists.")
    return filepath

# Download annotations files
box_csv_path = download_file(VAL_BOX_URL, "validation-annotations-bbox.csv")
meta_csv_path = download_file(VAL_METADATA_URL, "validation-images-with-rotation.csv")

In [ ]:
# Load files into pandas and filter
print("Loading annotations...")
df_box = pd.read_csv(box_csv_path)
df_meta = pd.read_csv(meta_csv_path)

# Filter for bounding boxes containing target classes
df_filtered = df_box[df_box["LabelName"].isin(TARGET_CLASSES.keys())].copy()
df_filtered["ClassName"] = df_filtered["LabelName"].map(TARGET_CLASSES)

# Grab 10 unique images for demonstration
unique_ids = df_filtered["ImageID"].unique()[:10]
df_meta_subset = df_meta[df_meta["ImageID"].isin(unique_ids)]

print(f"Found {len(unique_ids)} images for training/testing.")

In [ ]:
# Download images and collect labels
annotations_data = {}

for idx, row in df_meta_subset.iterrows():
    img_id = row["ImageID"]
    img_url = row["OriginalURL"]
    img_path = os.path.join(IMAGES_DIR, f"{img_id}.jpg")
    
    try:
        if not os.path.exists(img_path):
            urllib.request.urlretrieve(img_url, img_path)
        
        img_anns = df_filtered[df_filtered["ImageID"] == img_id]
        bboxes = []
        for _, ann in img_anns.iterrows():
            # Coordinate format in Open Images: [ymin, xmin, ymax, xmax] (normalized 0-1)
            bboxes.append({
                "class": ann["ClassName"],
                "box": [ann["YMin"], ann["XMin"], ann["YMax"], ann["XMax"]
            ]})
            
        annotations_data[f"{img_id}.jpg"] = {
            "width": int(row["OriginalWidth"]) if not pd.isna(row["OriginalWidth"]) else None,
            "height": int(row["OriginalHeight"]) if not pd.isna(row["OriginalHeight"]) else None,
            "detections": bboxes
        }
        print(f"Successfully downloaded image: {img_id}.jpg")
    except Exception as e:
        print(f"Failed to download {img_id}: {e}")

## Step 3: Visualize Annotations
Let's load one of the downloaded images and overlay the bounding box labels using Matplotlib.

In [ ]:
# Pick the first image from annotations
sample_filename = list(annotations_data.keys())[0]
sample_info = annotations_data[sample_filename]

img_path = os.path.join(IMAGES_DIR, sample_filename)
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

plt.figure(figsize=(10, 8))
ax = plt.gca()
plt.imshow(img)

# Draw each bounding box
for det in sample_info["detections"]:
    ymin, xmin, ymax, xmax = det["box"]
    # Convert normalized coordinates to absolute pixels
    box_x = xmin * w
    box_y = ymin * h
    box_w = (xmax - xmin) * w
    box_h = (ymax - ymin) * h
    
    rect = plt.Rectangle((box_x, box_y), box_w, box_h, fill=False, color='cyan', linewidth=3)
    ax.add_patch(rect)
    ax.text(box_x, box_y - 10, det["class"], bbox=dict(facecolor='cyan', alpha=0.8), fontsize=12, color='black')

plt.axis('off')
plt.show()

## Step 4: Load & Prepare Data for Training
We load all the images, resize them to `300x300` pixels (standard for SSD MobileNet), normalize pixel values, and format their classification/localization labels.

In [ ]:
IMG_SIZE = 300
CLASS_MAP = {"person": 0, "laptop": 1, "chair": 2, "book": 3, "table": 4}
NUM_CLASSES = len(CLASS_MAP)

x_train = []
y_class = []
y_bbox = []

for filename, data in annotations_data.items():
    img_path = os.path.join(IMAGES_DIR, filename)
    if not os.path.exists(img_path): continue
    
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    x_train.append(img / 255.0) # normalize
    
    # Select the primary object in image for regression mapping
    if len(data["detections"]) > 0:
        det = data["detections"][0]
        y_class.append(CLASS_MAP[det["class"]])
        y_bbox.append(det["box"])
    else:
        y_class.append(0)
        y_bbox.append([0.0, 0.0, 0.0, 0.0])

x_train = np.array(x_train, dtype=np.float32)
y_class = np.array(y_class, dtype=np.int32)
y_bbox = np.array(y_bbox, dtype=np.float32)

print(f"Dataset compiled: {len(x_train)} samples loaded.")

## Step 5: Transfer Learning Model Definition
We load a pre-trained **MobileNetV2** backbone (trained on ImageNet). We freeze the backbone layers to retain general visual features and add the custom prediction layers on top:
1. **Classification Head:** Outputs probability logits for the 5 custom Open Images classes.
2. **Bounding Box Regression Head:** Outputs 4 coordinates `[ymin, xmin, ymax, xmax]` representing the predicted object bounding box.

In [ ]:
def build_ssd_transfer_model():
    # Load MobileNetV2 features backbone
    backbone = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    backbone.trainable = False
    
    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    
    # Classification head output
    class_out = layers.Dense(NUM_CLASSES, activation="softmax", name="class_head")(x)
    
    # Regression head output
    bbox_out = layers.Dense(4, activation="sigmoid", name="bbox_head")(x)
    
    model = Model(inputs=backbone.input, outputs=[class_out, bbox_out])
    return model

model = build_ssd_transfer_model()
model.summary()

## Step 6: Custom Loss & Optimization
Object detection training utilizes a combined loss:
1. **Categorical Classification Loss:** Cross-entropy on the predicted class probabilities.
2. **Bounding Box Regression Loss (Smooth L1 Loss):** Measures the distance between predicted and ground truth coords.

In [ ]:
def multi_box_loss(y_true_cls, y_pred_cls, y_true_box, y_pred_box):
    # Sparse Categorical Cross-Entropy for class probabilities
    cls_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true_cls, y_pred_cls)
    cls_loss = tf.reduce_mean(cls_loss)
    
    # Smooth L1 Loss for bounding box regression
    mask = tf.cast(tf.reduce_sum(y_true_box, axis=-1) > 0, tf.float32)
    diff = tf.abs(y_true_box - y_pred_box)
    smooth_l1 = tf.where(diff < 1.0, 0.5 * tf.square(diff), diff - 0.5)
    box_loss = tf.reduce_sum(smooth_l1, axis=-1) * mask
    box_loss = tf.reduce_mean(box_loss)
    
    # Combined Loss (weighted sum)
    return cls_loss + 2.0 * box_loss, cls_loss, box_loss

## Step 7: Training Loop
We train the model weights of the output layers on our custom Open Images subset using Adam optimizer.

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
batch_size = 2

print("Starting training output layers...")
for epoch in range(EPOCHS):
    epoch_total_loss = 0
    num_batches = int(np.ceil(len(x_train) / batch_size))
    
    for step in range(num_batches):
        start_idx = step * batch_size
        end_idx = min(start_idx + batch_size, len(x_train))
        
        x_batch = x_train[start_idx:end_idx]
        y_cls_batch = y_class[start_idx:end_idx]
        y_box_batch = y_bbox[start_idx:end_idx]
        
        with tf.GradientTape() as tape:
            pred_cls, pred_box = model(x_batch, training=True)
            total_loss, cls_l, box_l = multi_box_loss(y_cls_batch, pred_cls, y_box_batch, pred_box)
            
        grads = tape.gradient(total_loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        epoch_total_loss += total_loss.numpy()
        
    print(f"Epoch {epoch+1}/{EPOCHS} completed - Loss: {epoch_total_loss/num_batches:.4f}")

## Step 8: Test Inference
Let's pass a test image through our fine-tuned SSD and visualize the predicted output classification and localization bounding boxes!

In [ ]:
# Grab a test sample
test_idx = 0
test_img = np.expand_dims(x_train[test_idx], axis=0)

# Run inference
pred_cls, pred_box = model.predict(test_img)

class_id = np.argmax(pred_cls[0])
confidence = pred_cls[0][class_id]
bbox = pred_box[0]

inverted_class_map = {v: k for k, v in CLASS_MAP.items()}
class_label = inverted_class_map[class_id]

print(f"Predicted Class: {class_label} (Confidence: {confidence*100:.1f}%)")
print(f"Predicted Bounding Box: {bbox}")

In [ ]:
# Render predicted bounding box
display_img = (x_train[test_idx] * 255.0).astype(np.uint8)
ymin, xmin, ymax, xmax = bbox

h, w = display_img.shape[:2]
px_x = int(xmin * w)
px_y = int(ymin * h)
px_w = int((xmax - xmin) * w)
px_h = int((ymax - ymin) * h)

plt.figure(figsize=(8, 6))
ax = plt.gca()
plt.imshow(display_img)

rect = plt.Rectangle((px_x, px_y), px_w, px_h, fill=False, color='red', linewidth=3)
ax.add_patch(rect)
ax.text(px_x, px_y - 10, f"{class_label} ({confidence*100:.1f}%)", 
        bbox=dict(facecolor='red', alpha=0.8), fontsize=12, color='white')

plt.axis('off')
plt.show()